# Compact Vision: CIFAR-10 → PTQ (int8) → Benchmark → Demo Artifacts

**What this notebook does (offline, reproducible):**
1) Train a *baseline* model on CIFAR-10 (uses GPU if available).
2) Post-Training Quantization (FX/PTQ) on CPU → export TorchScript `compressed.ts` (int8, CPU target).
3) Evaluate accuracy (baseline eager on CPU/GPU + compressed TS on CPU).
4) Benchmark latency/throughput (baseline eager on CPU + compressed TS on CPU).
5) Summarize results → `outputs/results.csv` and 2 plots.

Artifacts (contract):
- `outputs/baseline.pt`, `outputs/compressed.ts`
- `outputs/acc_resnet18_eager_cpu.json`, `outputs/acc_resnet18_ts_cpu.json`
- `outputs/bench_resnet18_eager_cpu.json`, `outputs/bench_resnet18_ts_cpu.json`
- `outputs/results.csv`, `outputs/plot_acc_vs_latency.png`, `outputs/plot_acc_vs_size.png`


In [5]:
import sys, os, json, time, math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets

sys.path.insert(0, os.path.abspath(".."))

from utils import contract
from utils.device import pick_device, cpu_device, device_name
from utils.determinism import set_deterministic
from utils.data_transforms import train_transform_cifar10, test_transform_cifar10, calib_transform_cifar10
from utils.models import build_model, params_millions

# Repro by default (can switch to performance later if you want)
set_deterministic(42)

# Ensure outputs dir exists
contract.OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Sanity: local data must exist
assert contract.CIFAR10_DIR.exists(), f"Missing dataset at {contract.CIFAR10_DIR}. Unzip CIFAR-10 there."

print("Data OK:", contract.CIFAR10_DIR)
print("Outputs dir:", contract.OUTPUTS_DIR.resolve())
print("CUDA available:", torch.cuda.is_available())


Data OK: C:\Users\khal6952\Documents\GitHub\compact-ml-bench\data\cifar-10-batches-py
Outputs dir: C:\Users\khal6952\Documents\GitHub\compact-ml-bench\outputs
CUDA available: True


In [6]:
def make_cifar_loaders(data_root: Path, batch: int = 256, workers: int = 2):
    train_set = datasets.CIFAR10(root=str(data_root), train=True,  download=False, transform=train_transform_cifar10())
    test_set  = datasets.CIFAR10(root=str(data_root), train=False, download=False, transform=test_transform_cifar10())

    train_loader = DataLoader(train_set, batch_size=batch, shuffle=True,  num_workers=workers, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=256,  shuffle=False, num_workers=workers, pin_memory=True)
    return train_loader, test_loader

def make_calib_loader(data_root: Path, batch: int = 128, workers: int = 2, max_batches: int = 8):
    ds = datasets.CIFAR10(root=str(data_root), train=True, download=False, transform=calib_transform_cifar10())
    dl = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=workers, pin_memory=False)
    # Yield only a handful of batches for PTQ calibration
    def _gen():
        for i, (x, _) in enumerate(dl):
            if i >= max_batches:
                break
            yield x
    return _gen()

train_loader, test_loader = make_cifar_loaders(contract.DATA_DIR, batch=256, workers=2)
calib_stream = make_calib_loader(contract.DATA_DIR, batch=128, workers=2, max_batches=8)

len_train = len(train_loader.dataset)
len_test  = len(test_loader.dataset)
len_train, len_test


(50000, 10000)

In [7]:
EPOCHS   = 2                 # keep small for CPU-friendliness
ARCH     = contract.DEFAULT_ARCH
DEVICE   = pick_device(prefer_cuda=True)   # GPU if available, else CPU
LR       = 0.1
WD       = 5e-4

model = build_model(ARCH, num_classes=contract.NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

use_amp = (DEVICE.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def evaluate_top1_eager(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return 100.0 * correct / total

# Train
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * x.size(0)
    scheduler.step()

    train_loss = running / len_train
    test_acc = evaluate_top1_eager(model, test_loader, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | test_acc={test_acc:.2f}% on {device_name(DEVICE)}")

# Save baseline checkpoint (state_dict)
torch.save({"arch": ARCH, "state_dict": model.state_dict()}, contract.BASELINE_CKPT)
print("Saved baseline:", contract.BASELINE_CKPT.resolve())


Epoch 1/2 | train_loss=2.1108 | test_acc=35.44% on cuda
Epoch 2/2 | train_loss=1.5166 | test_acc=51.23% on cuda
Saved baseline: C:\Users\khal6952\Documents\GitHub\compact-ml-bench\outputs\baseline.pt


In [8]:
# PTQ is a CPU flow; set engine and rebuild model on CPU
import torch.ao.quantization as tq
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

    
torch.backends.quantized.engine = "fbgemm"
example_inputs = torch.randn(1, 3, 32, 32)

# Rebuild same arch on CPU and load baseline weights
m_fp32 = build_model(ARCH, num_classes=contract.NUM_CLASSES).cpu()
state = torch.load(contract.BASELINE_CKPT, map_location="cpu")
m_fp32.load_state_dict(state["state_dict"], strict=True)
m_fp32.eval()

# QConfig mapping (default per-backend)
qconfig_mapping = tq.get_default_qconfig_mapping("fbgemm")

# Prepare FX graph for quantization
prepared = prepare_fx(m_fp32, qconfig_mapping, example_inputs)

# Calibration: run a few CPU batches through observers
with torch.inference_mode():
    for i, x in enumerate(make_calib_loader(contract.DATA_DIR, batch=128, workers=0, max_batches=8)):
        prepared(x)
# Convert to quantized model
quantized = convert_fx(prepared)
quantized.eval()

# Export TorchScript int8 (trace is robust for quantized graphs)
ts = torch.jit.trace(quantized, example_inputs)
torch.jit.save(ts, contract.COMPRESSED_TS)
print("Saved compressed TS:", contract.COMPRESSED_TS.resolve())


RuntimeError: quantized engine FBGEMM is not supported

In [ ]:
def evaluate_ts(ts_module: torch.jit.ScriptModule, loader: DataLoader) -> float:
    correct = 0
    total = 0
    with torch.inference_mode():
        for x, y in loader:
            x_cpu, y_cpu = x.cpu(), y.cpu()
            logits = ts_module(x_cpu)
            pred = logits.argmax(dim=1)
            correct += (pred == y_cpu).sum().item()
            total += y_cpu.numel()
    return 100.0 * correct / total

# Eager accuracy on CPU (for apples-to-apples with TS)
m_cpu = build_model(ARCH, num_classes=contract.NUM_CLASSES).cpu()
m_cpu.load_state_dict(torch.load(contract.BASELINE_CKPT, map_location="cpu")["state_dict"])
acc_eager_cpu = evaluate_top1_eager(m_cpu, test_loader, cpu_device())
with open(contract.ACC_EAGER_JSON, "w") as f:
    json.dump({"variant":"resnet18_eager_cpu","arch":ARCH,"acc_top1":acc_eager_cpu}, f, indent=2)
print("Saved:", contract.ACC_EAGER_JSON.name, f"{acc_eager_cpu:.2f}%")

# TS accuracy (CPU)
ts_module = torch.jit.load(contract.COMPRESSED_TS, map_location="cpu")
acc_ts_cpu = evaluate_ts(ts_module, test_loader)
with open(contract.ACC_TS_JSON, "w") as f:
    json.dump({"variant":"resnet18_ts_cpu","arch":ARCH,"acc_top1":acc_ts_cpu}, f, indent=2)
print("Saved:", contract.ACC_TS_JSON.name, f"{acc_ts_cpu:.2f}%")

# (Optional) also compute eager accuracy on GPU for your own curiosity
if torch.cuda.is_available():
    acc_eager_gpu = evaluate_top1_eager(model, test_loader, DEVICE)
    print(f"[Optional] Eager accuracy on GPU: {acc_eager_gpu:.2f}%")


In [ ]:
def _percentiles_ms(samples_ms):
    a = np.array(samples_ms, dtype=np.float64)
    return {
        "mean_ms": float(a.mean()),
        "std_ms":  float(a.std(ddof=1)) if a.size > 1 else 0.0,
        "p50_ms":  float(np.percentile(a, 50)),
        "p90_ms":  float(np.percentile(a, 90)),
        "p99_ms":  float(np.percentile(a, 99)),
        "samples": int(a.size),
    }

def _bench_model(run_fn, batch_sizes=(1, 8), warmup=30, repeat=200):
    results = {}
    for b in batch_sizes:
        # warmup
        for _ in range(warmup):
            run_fn(b)
        # timed
        times_ms = []
        for _ in range(repeat):
            t0 = time.perf_counter()
            run_fn(b)
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000.0)
        stats = _percentiles_ms(times_ms)
        # throughput approx: images / mean_seconds
        img_s = (b / (stats["mean_ms"] / 1000.0)) if stats["mean_ms"] > 0 else 0.0
        results[str(b)] = {"stats": stats, "img_s": float(img_s)}
    return results

def _artifact_size_mb(p: Path) -> float:
    return round(p.stat().st_size / (1024*1024), 3)


In [ ]:
# Build model on CPU for the contract CPU bench
m_bench_cpu = build_model(ARCH, num_classes=contract.NUM_CLASSES).cpu()
m_bench_cpu.load_state_dict(torch.load(contract.BASELINE_CKPT, map_location="cpu")["state_dict"])
m_bench_cpu.eval()

@torch.inference_mode()
def run_eager_cpu(batch: int):
    x = torch.randn(batch, 3, 32, 32, device="cpu")
    _ = m_bench_cpu(x)

bench_eager = _bench_model(run_eager_cpu, batch_sizes=(1, 8), warmup=60, repeat=300)

bench_eager_json = {
    "schema": "bench.v2",
    "variant": "resnet18_eager_cpu",
    "arch": ARCH,
    "device": "cpu",
    "threads": torch.get_num_threads(),
    "params_millions": float(params_millions(m_bench_cpu)),
    "size_mb": float(_artifact_size_mb(contract.BASELINE_CKPT)),
    "batch_sizes": {k: v["stats"] for k, v in bench_eager.items()},
    "throughput_img_s": {k: v["img_s"] for k, v in bench_eager.items()},
}
with open(contract.BENCH_EAGER_JSON, "w") as f:
    json.dump(bench_eager_json, f, indent=2)
print("Saved:", contract.BENCH_EAGER_JSON.name)


Saved: bench_resnet18_eager_cpu.json


In [ ]:
ts_bench = torch.jit.load(contract.COMPRESSED_TS, map_location="cpu")
ts_bench.eval()

@torch.inference_mode()
def run_ts_cpu(batch: int):
    x = torch.randn(batch, 3, 32, 32, device="cpu")
    _ = ts_bench(x)

bench_ts = _bench_model(run_ts_cpu, batch_sizes=(1, 8), warmup=60, repeat=300)

bench_ts_json = {
    "schema": "bench.v2",
    "variant": "resnet18_ts_cpu",
    "arch": ARCH,
    "device": "cpu",
    "threads": torch.get_num_threads(),
    "params_millions": None,  # TS graph; param count less meaningful here
    "size_mb": float(_artifact_size_mb(contract.COMPRESSED_TS)),
    "batch_sizes": {k: v["stats"] for k, v in bench_ts.items()},
    "throughput_img_s": {k: v["img_s"] for k, v in bench_ts.items()},
}
with open(contract.BENCH_TS_JSON, "w") as f:
    json.dump(bench_ts_json, f, indent=2)
print("Saved:", contract.BENCH_TS_JSON.name)


ValueError: The provided filename C:\Users\khal6952\Documents\GitHub\compact-ml-bench\outputs\compressed.ts does not exist